1. Configs

In [19]:
import sys
import os

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.append(project_root)

from pipeline.context import ClaimContext
from pipeline.pipeline_factory import pipeline_rule_verdict, pipeline_majority_verdict
from evaluation.evaluate_predictions import evaluate

2. Load dataset

In [20]:
import json
from pathlib import Path

def load_averitec_split(split="dev"):
    path = Path("../datasets/averitec") / f"{split}.json"
    
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    return data

3. Run pipeline

In [21]:
data = load_averitec_split("dev")[:1] 

pipeline = pipeline_rule_verdict()

results = []

for item in data:
    context = ClaimContext(
        claim_id=item["id"] if "id" in item else None,
        claim_text=item["claim"],
        claim_date=item.get("claim_date")
    )
    
    result = pipeline.run(context)

    results.append({
    "claim": item["claim"],
    "prediction": result.verdict,
    "gold_label": item.get("label"),

    "pipeline": {
        "questions": getattr(result, "questions", []),
        "qa_pairs": getattr(result, "qa_pairs", []),
        "evidence": result.evidence,
        "stances": result.stances
    }
})

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 714.67it/s, Materializing param=classifier.weight]                                    
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


4. Save results

In [22]:
evidence_with_stance = []

for e, s in zip(result.evidence, result.stances):
    evidence_with_stance.append({
        "text": e["text"],
        "bm25_score": e.get("bm25_score"),
        "rerank_score": e.get("rerank_score"),
        "stance": s["label"]
    })

import json

with open("results_dev.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

6. Ground truth compare

In [23]:
correct = 0

for item, pred in zip(data, results):
    if item["label"].lower() == pred["prediction"].lower():
        correct += 1

print("Accuracy:", correct / len(results))

Accuracy: 1.0


In [24]:
evaluate("results_dev.json", "../datasets/averitec/dev.json")


===== Evaluation =====

Accuracy: 1.0

Per-class metrics:

refuted
  Precision: 1.0
  Recall:    1.0
  F1:        1.0

Macro F1: 1.0
